# OAB Pipeline — Benchmark de Modelos de Linguagem

Avalia modelos open-source em questões discursivas e objetivas da OAB.

**Pipeline:**
1. Setup — monta Drive, clona repositório
2. Preparo dos dados — carrega datasets e gera CSVs base
3. Respostas — modelos candidatos respondem as questões
4. Curadoria — modelo curador classifica as questões
5. Avaliação — modelo juiz avalia as respostas discursivas
6. Resultados — accuracy objetivas + benchmark discursivas

## 1. Setup

In [ ]:
# Monta o Google Drive — garante persistência entre sessões
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clona o repositório (pula se já existir)
import os
if not os.path.exists('/content/oab_pipeline'):
    !git clone https://github.com/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1 /content/oab_pipeline

%cd /content/oab_pipeline

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets pandas tqdm

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
from google.colab import userdata
from huggingface_hub import whoami

# Reprodutibilidade
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Autenticação Hugging Face
import os
token = userdata.get('HF_TOKEN')
if not token:
    raise ValueError("HF_TOKEN não encontrado. Configure nas Secrets do Colab.")
os.environ['HF_TOKEN'] = token
print(whoami())

# GPU obrigatória
if not torch.cuda.is_available():
    raise RuntimeError("GPU não disponível. Ative em Ambiente de execução > Alterar tipo de hardware > T4.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Preparo dos dados

In [ ]:
import json
from datasets import load_dataset
from config import (
    QUESTOES_DISCURSIVAS_CSV, QUESTOES_OBJETIVAS_CSV,
    DISC_SLICE_START, DISC_SLICE_END,
    OBJ_SLICE_START, OBJ_SLICE_END,
)
from data_utils import salvar_csv

# --- Discursivas ---
dataset = load_dataset('maritaca-ai/oab-bench', name='questions')
df_disc_raw = pd.DataFrame(dataset['train']).iloc[DISC_SLICE_START:DISC_SLICE_END].reset_index(drop=True)

def montar_texto_questao(row):
    enunciado = str(row['statement']).strip()
    itens = row.get('turns', [])
    if isinstance(itens, list):
        itens = [str(i).strip() for i in itens if str(i).strip()]
    else:
        itens = []
    if not itens:
        return enunciado
    itens_fmt = '\n\n'.join(f'{n}. {item}' for n, item in enumerate(itens, 1))
    return f'{enunciado}\n\n{itens_fmt}'

df_disc = pd.DataFrame({
    'question_id': df_disc_raw['question_id'],
    'dataset': 'J1',
    'tipo_questao': 'discursiva',
    'categoria_dataset': df_disc_raw['category'],
    'texto_questao': df_disc_raw.apply(montar_texto_questao, axis=1),
})
salvar_csv(df_disc, QUESTOES_DISCURSIVAS_CSV)
display(df_disc.head())

# --- Objetivas ---
dataset_obj = load_dataset('eduagarcia/oab_exams')
df_obj_raw = pd.DataFrame(dataset_obj['train']).iloc[OBJ_SLICE_START:OBJ_SLICE_END].reset_index(drop=True)

def normalizar_choices(choices):
    return json.dumps(dict(zip(choices.get('label', []), choices.get('text', []))), ensure_ascii=False)

df_obj = pd.DataFrame({
    'question_id': df_obj_raw['id'],
    'dataset': 'J2',
    'tipo_questao': 'objetiva',
    'numero_questao': df_obj_raw['question_number'],
    'categoria_dataset': df_obj_raw['question_type'],
    'texto_questao': df_obj_raw['question'],
    'alternativas_json': df_obj_raw['choices'].apply(normalizar_choices),
    'gabarito_oficial': df_obj_raw['answerKey'],
})
salvar_csv(df_obj, QUESTOES_OBJETIVAS_CSV)
display(df_obj.head())

## 3. Respostas dos candidatos

In [ ]:
from tqdm import tqdm
from config import MODELOS_CANDIDATOS, RESPOSTAS_DISCURSIVAS_CSV, RESPOSTAS_OBJETIVAS_CSV
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_resposta_discursiva, gerar_resposta_objetiva

questoes_disc = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
questoes_obj  = carregar_csv(QUESTOES_OBJETIVAS_CSV)

respostas_disc = []
respostas_obj  = []

# Resume: carrega o que já foi processado nas sessões anteriores
ids_disc_feitos = set()
ids_obj_feitos  = set()
if RESPOSTAS_DISCURSIVAS_CSV.exists():
    df_ex = carregar_csv(RESPOSTAS_DISCURSIVAS_CSV)
    respostas_disc = df_ex.to_dict('records')
    ids_disc_feitos = set(zip(df_ex['question_id'], df_ex['modelo']))
if RESPOSTAS_OBJETIVAS_CSV.exists():
    df_ex = carregar_csv(RESPOSTAS_OBJETIVAS_CSV)
    respostas_obj = df_ex.to_dict('records')
    ids_obj_feitos = set(zip(df_ex['question_id'], df_ex['modelo']))

for nome_modelo, path_modelo in MODELOS_CANDIDATOS.items():
    print(f'\nModelo: {nome_modelo}')
    model, tok = load_model(path_modelo)

    print('  → Discursivas...')
    for _, row in tqdm(questoes_disc.iterrows(), total=len(questoes_disc)):
        if (row['question_id'], nome_modelo) in ids_disc_feitos:
            continue
        respostas_disc.append(gerar_resposta_discursiva(model, tok, row.to_dict(), nome_modelo))

    print('  → Objetivas...')
    for _, row in tqdm(questoes_obj.iterrows(), total=len(questoes_obj)):
        if (row['question_id'], nome_modelo) in ids_obj_feitos:
            continue
        respostas_obj.append(gerar_resposta_objetiva(model, tok, row.to_dict(), nome_modelo))

    unload(model, tok)

salvar_csv(pd.DataFrame(respostas_disc), RESPOSTAS_DISCURSIVAS_CSV)
salvar_csv(pd.DataFrame(respostas_obj),  RESPOSTAS_OBJETIVAS_CSV)

## 4. Curadoria

In [ ]:
from tqdm import tqdm
from config import MODELO_CURADOR, CURADORIA_DISCURSIVAS_CSV, CURADORIA_OBJETIVAS_CSV
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_curadoria

questoes_disc = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
questoes_obj  = carregar_csv(QUESTOES_OBJETIVAS_CSV)

model, tok = load_model(MODELO_CURADOR)

cur_disc = [gerar_curadoria(model, tok, row.to_dict(), 'discursiva')
            for _, row in tqdm(questoes_disc.iterrows(), total=len(questoes_disc))]

cur_obj  = [gerar_curadoria(model, tok, row.to_dict(), 'objetiva')
            for _, row in tqdm(questoes_obj.iterrows(), total=len(questoes_obj))]

unload(model, tok)

salvar_csv(pd.DataFrame(cur_disc), CURADORIA_DISCURSIVAS_CSV)
salvar_csv(pd.DataFrame(cur_obj),  CURADORIA_OBJETIVAS_CSV)
display(pd.DataFrame(cur_disc).head())

## 5. Avaliação discursiva

In [ ]:
from tqdm import tqdm
from config import MODELO_JUIZ, AVALIACAO_DISCURSIVAS_CSV
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_avaliacao_discursiva

questoes_disc  = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
respostas_disc = carregar_csv(RESPOSTAS_DISCURSIVAS_CSV)
curadoria_disc = carregar_csv(CURADORIA_DISCURSIVAS_CSV)

model, tok = load_model(MODELO_JUIZ)
avaliacoes = []

for _, resp_row in tqdm(respostas_disc.iterrows(), total=len(respostas_disc)):
    resp_row = resp_row.to_dict()
    q_match = questoes_disc.loc[questoes_disc['question_id'] == resp_row['question_id']]
    c_match = curadoria_disc.loc[curadoria_disc['question_id'] == resp_row['question_id']]

    if q_match.empty or c_match.empty:
        print(f"Questão/curadoria não encontrada: {resp_row['question_id']}")
        continue

    avaliacoes.append(gerar_avaliacao_discursiva(
        model, tok,
        q_match.iloc[0].to_dict(),
        resp_row,
        c_match.iloc[0].to_dict(),
    ))

unload(model, tok)
salvar_csv(pd.DataFrame(avaliacoes), AVALIACAO_DISCURSIVAS_CSV)
display(pd.DataFrame(avaliacoes).head())

## 6. Resultados

In [ ]:
from config import ACCURACY_MODELOS_CSV, BENCHMARK_DISCURSIVAS_CSV
from data_utils import carregar_csv, salvar_csv

# Accuracy — objetivas
df_obj_res = carregar_csv(RESPOSTAS_OBJETIVAS_CSV)
accuracy = df_obj_res.groupby('modelo')['correto'].mean().reset_index(name='accuracy')
salvar_csv(accuracy, ACCURACY_MODELOS_CSV)
display(accuracy)

# Benchmark — discursivas por área
df_aval  = carregar_csv(AVALIACAO_DISCURSIVAS_CSV)
df_cur   = carregar_csv(CURADORIA_DISCURSIVAS_CSV)

df_merged = df_aval.merge(
    df_cur[['question_id', 'area_especialidade_equipe']],
    on='question_id', how='left'
)

df_benchmark = (
    df_merged
    .groupby(['area_especialidade_equipe', 'modelo_candidato'])['nota']
    .mean()
    .reset_index()
    .pivot(index='area_especialidade_equipe', columns='modelo_candidato', values='nota')
    .round(2)
)
df_benchmark.loc['Mean'] = df_benchmark.mean().round(2)
df_benchmark.index.name  = 'Área de Especialidade'
df_benchmark.columns.name = None

salvar_csv(df_benchmark, BENCHMARK_DISCURSIVAS_CSV)
display(df_benchmark)